# 02 · Features and RQ1 (acoustic feature discrimination)

Extracts the 64 clip-level features and log-mel spectrograms, re-estimates the intra-class correlation for the sample-size check, and answers **RQ1**: which features differ between human and synthetic speech, and by how much.

H₀: the feature distribution is the same for human and synthetic clips. H₁: it differs.

Runtime: feature extraction (pYIN pitch tracking) ≈ 30–60 min on Colab for all 11,570 clips plus 4,370 telephone copies.

In [ ]:
# --- Setup: works on Google Colab and locally -------------------------------
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ.setdefault("VS_DATA_ROOT", "/content/drive/MyDrive/audio-detection-human-vs-ai")
    if not os.path.exists("/content/audio-detection-human-vs-ai"):
        !git clone -q https://github.com/abhilash-u/audio-detection-human-vs-ai /content/audio-detection-human-vs-ai
    %cd /content/audio-detection-human-vs-ai
    !pip install -q -r requirements.txt
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import numpy as np, pandas as pd
from src import config as C
C.ensure_dirs()
print("Data folder:", C.DATA_ROOT)

In [ ]:
from src import features as F
sample = pd.read_csv(C.META_DIR / "split_assignments.csv")
X, L = F.build_feature_table(sample.clip_id.tolist(), C.CLEAN_DIR)
X.to_csv(C.FEAT_DIR / "features_clean.csv", index=False)
np.save(C.FEAT_DIR / "logmel_clean.npy", L)
test = sample[sample.split.str.startswith("test")]
Xp, Lp = F.build_feature_table(test.clip_id.tolist(), C.PHONE_DIR)
Xp.to_csv(C.FEAT_DIR / "features_telephone.csv", index=False)
np.save(C.FEAT_DIR / "logmel_telephone.npy", Lp)
X.shape, L.shape, Xp.shape

## Re-estimate ρ and confirm the sample size

In [ ]:
from src import sample_size as S
df = sample.merge(X, on="clip_id")
pool = df[df.split.isin(["training", "validation"])]
icc = pd.Series({f: S.estimate_icc(pool[f].values, pool.speaker_id.values) for f in F.FEATURES})
rho_hat = float(icc.median())
print(f"Median ICC across features: {rho_hat:.3f}  (planning value 0.05)")
table, final_n = S.summary(max(rho_hat, 0.0))
display(table); print("Final N with estimated rho:", final_n)

## RQ1: tests and effect sizes (training + validation pool)
With thousands of clips nearly every feature will be significant, so conclusions rest on effect sizes. p-values are adjusted with Benjamini–Hochberg.

In [ ]:
from src import evaluation as E
res = E.feature_tests(pool, F.FEATURES)
display(res.head(15).round(4))

## Speaker-cluster bootstrap 95% CIs for Cohen's d (top 15 features)

In [ ]:
rows = []
for f in res.feature.head(15):
    est, lo, hi = E.cluster_bootstrap(pool, lambda d, f=f: E.cohens_d(d.loc[d.label == 1, f].values, d.loc[d.label == 0, f].values), n_boot=1000)
    rows.append({"feature": f, "cohens_d": est, "ci_low": lo, "ci_high": hi})
ci = pd.DataFrame(rows)
res.merge(ci[["feature", "ci_low", "ci_high"]], on="feature", how="left").to_csv(C.REPORT_DIR / "rq1_feature_effects.csv", index=False)
ci.round(3)

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7, 5))
c = ci.iloc[::-1]
ax.errorbar(c.cohens_d, range(len(c)), xerr=[c.cohens_d - c.ci_low, c.ci_high - c.cohens_d], fmt="o", color="#2F4B66")
ax.axvline(0, color="grey", lw=0.8); ax.set_yticks(range(len(c))); ax.set_yticklabels(c.feature)
ax.set_xlabel("Cohen's d (synthetic − human), 95% cluster-bootstrap CI"); ax.set_title("RQ1: most discriminative features")
plt.tight_layout(); plt.savefig(C.REPORT_DIR / "rq1_effect_sizes.png", dpi=200); plt.show()